# STG 3 — Filtro de títulos sin valor semántico

**Objetivo**: eliminar de `df_consolidado` los registros cuyo `titulo_base` no describe
el contenido temático de una ley (mociones, habilitaciones, expedientes sin descripción, etc.).

**Entrada**: `data/df_consolidado.csv`  
**Salida**: `data/df_modelado.csv`

> ⚠️ Este notebook tiene dos celdas críticas separadas:
> - **Celda de INSPECCIÓN**: muestra qué se va a filtrar. No elimina nada.
> - **Celda de APLICACIÓN**: aplica el filtro y guarda. Ejecutar solo después de revisar la inspección.

In [ ]:
import pandas as pd
import re

df = pd.read_csv('../data/df_consolidado.csv')

print('Columnas:', df.columns.tolist())
print('Shape:', df.shape)
print('\nPrimeras filas:')
df.head()

## Función de filtrado

Define los patrones que identifican títulos sin valor semántico.
Cada grupo corresponde a una categoría documentada en la spec 002.

In [ ]:
PATRONES_SIN_VALOR = [
    # Habilitación de tratamiento (procedimiento parlamentario, no ley sustantiva)
    # Ej: HABILITACIÓN DEL TRATAMIENTO EXPTE. 9-PE-2025
    r'^HABILITACI[OÓ]N DEL TRATAMIENTO',
    r'^HABILITACION DEL TRATAMIENTO',
    r'^HABILITACI[OÓ]N DE TRATAMIENTO',
    r'^HABILITACI[OÓ]N PROYECTOS DE',

    # Insistencia de ley (solo número, sin descripción)
    # Ej: INSISTENCIA PROYECTO DE LEY 27.795
    r'^INSISTENCIA PROYECTO DE LEY\s+[\d\.]+\s*$',

    # Moción de emplazamiento (procedimiento, no ley)
    r'^MOCI[OÓ]N DE EMPLAZAMIENTO',
    r'^MOCION DE EMPLAZAMIENTO',
    r'^Moci[oó]n de emplazamiento',

    # Moción de reconsideración (procedimiento)
    r'^MOCION DE RECONSIDERACI[OÓ]N',
    r'^MOCI[OÓ]N DE RECONSIDERACI[OÓ]N',
    r'^Moci[oó]n de reconsideraci[oó]n',

    # Moción de orden (procedimiento parlamentario)
    # Ej: Moción de Orden solicitada por el Diputado CARMONA
    r'^Moci[oó]n de Orden',
    r'^MOCI[OÓ]N DE ORDEN',

    # Moción genérica solicitada por diputado
    r'^MOCION SOLICITADA',
    r'^mocion solicitada',

    # Pedido de preferencia / tratamiento sobre tablas (procedimiento)
    # Ej: Pedido de Preferencia solicitado por el Diputado PITROLA
    r'^Pedido de [Pp]referencia',
    r'^Pedido de [Tt]rat(amiento)?\s+sobre\s+[Tt]abla',

    # Apartamiento del reglamento
    # Ej: APARTAMIENTO DEL REGLAMENTO SOLICITADO POR LA DIP. MATZEN
    r'^APARTAMIENTO DEL REGLAMENTO',

    # Lista de órdenes del día sin descripción temática
    # Ej: OD 86 - 89 - 90 - 92 ...  /  od 386 - od 385 - od 368
    r'^OD\s+\d+\s*[-–]\s*\d+',
    r'^od\s+\d+\s*[-–]\s*od\s+\d+',

    # Plan de labor (agenda de sesión, no ley)
    r'^PLAN DE LABOR$',

    # Votación en general sin título propio de ley
    r'^VOTACI[OÓ]N EN GRAL\.',

    # Pliegos de magistrados de la Corte Suprema
    # (son designaciones de personas, no leyes con contenido temático)
    r'^Exp\.\s+7835-D-00',
]
# Nota: los patrones para EXPTE./EXP./Expediente solos se manejan por _tiene_descripcion().


def _tiene_descripcion(titulo):
    """
    Devuelve True si hay texto descriptivo después del número de expediente.
    Maneja dos formatos:
      - NNNN-XX-NNNN  (ej: 498-D-21, 0089-S-2020)
      - XX-NNNN[-NNNN] (ej: PE-76-2024, JGM-20)
    """
    # Quita el prefijo Exp./Expte./Expediente/Expedientes
    sin_prefijo = re.sub(r'^Exp(ediente|te)?s?\.?\s+', '', titulo.strip(), flags=re.IGNORECASE)
    # Intenta quitar número formato NNNN-XX-NNNN
    sin_numero = re.sub(r'^\d+\s*[-–]\s*[A-Za-z]+\s*[-–]\s*\d+\s*', '', sin_prefijo)
    # Si no matcheó, intenta formato código-número (PE-76-2024, JGM-20)
    if sin_numero == sin_prefijo:
        sin_numero = re.sub(r'^[A-Za-z]+-\d+(-\d+)?\s*$', '', sin_prefijo)
    # Quita separadores iniciales (punto, guión, barra, espacio)
    sin_separador = re.sub(r'^[\s\.\-\/]+', '', sin_numero.strip())
    # "y otros" solo no es descripción temática
    sin_separador = re.sub(r'^y\s+otros?\s*$', '', sin_separador.strip(), flags=re.IGNORECASE)
    # "Orden del Día NNN" solo tampoco es descripción temática
    sin_separador = re.sub(r'^Orden del D[íi]a\s+\d+\s*$', '', sin_separador.strip(), flags=re.IGNORECASE)
    # "O.D. NNN" solo tampoco es descripción temática
    sin_separador = re.sub(r'^O\.?\s*D\.?\s+\d+\s*$', '', sin_separador.strip(), flags=re.IGNORECASE)
    return len(sin_separador.strip()) > 5


def es_sin_valor(titulo):
    """Devuelve True si el título no tiene valor semántico para el modelo."""
    t = str(titulo).strip()

    for patron in PATRONES_SIN_VALOR:
        if re.search(patron, t, re.IGNORECASE):
            return True

    # Regla especial: Exp./Expte./EXPTE./Expediente/EXP. con número solo
    # Conserva: "Expte. 0089-S-2020. DE LEY. CONVENIO DE TRANSFERENCIA PROGRESIVA A CABA"
    # Conserva: "Exp. 498-D-21 - Ayuda Transportistas Escolares"
    # Descarta:  "EXPTE. 5297-D-2025"
    # Descarta:  "Expediente 4259-D-2019"
    # Descarta:  "Exp. 2681-D-06 - Orden del Día 798"
    if re.match(r'^Exp(ediente|te)?s?\.?\s+', t, re.IGNORECASE):
        if not _tiene_descripcion(t):
            return True

    return False


print('Función es_sin_valor() definida.')

# Prueba rápida con casos conocidos
casos_prueba = [
    # (titulo, debe_filtrar)
    ('EXPTE. 5297-D-2025',                                                        True),
    ('INSISTENCIA PROYECTO DE LEY 27.795',                                        True),
    ('HABILITACIÓN DEL TRATAMIENTO EXPTE. 9-PE-2025',                             True),
    ('MOCIÓN DE EMPLAZAMIENTO SOLICITADA POR EL DIP. FERRARO',                    True),
    ('PLAN DE LABOR',                                                             True),
    ('OD 86 - 89 - 90 -92 - 103',                                                True),
    ('Exp. 3-PE-09',                                                              True),
    # Nuevos casos — grupos 1, 6, 9 identificados en STG_4
    ('Expediente 4259-D-2019',                                                    True),
    ('Expediente 5339-D-2018 - O.D. 1054',                                        True),
    ('Exp. 2681-D-06 - Orden del Día 798',                                        True),
    ('Exp. 101-S-06 - Orden del Día 667',                                         True),
    ('Moción de Orden solicitada por el Diputado CARMONA, Guillermo',             True),
    ('Pedido de Preferencia solicitado por el Diputado PITROLA, Néstor',          True),
    # Casos borderline: NO deben filtrarse
    ('Expte. 0089-S-2020. DE LEY. CONVENIO DE TRANSFERENCIA PROGRESIVA A CABA',  False),
    ('Exp. 498-D-21 - Ayuda Transportistas Escolares',                            False),
    ('Exp. 2183-D-09 -  Impuestos Internos',                                      False),
    # O.D. con descripción: NO deben filtrarse
    ('O.D. 923 - EMERGENCIA SANITARIA DE LA SALUD PEDIÁTRICA',                    False),
    ('O.D. 721 - LEY DE FICHA LIMPIA.',                                           False),
]

errores = 0
for titulo, esperado in casos_prueba:
    resultado = es_sin_valor(titulo)
    estado = '✓' if resultado == esperado else '✗ ERROR'
    if resultado != esperado:
        errores += 1
    print(f'{estado}  filtrar={resultado}  |  {titulo[:75]}')

print(f'\n{len(casos_prueba) - errores}/{len(casos_prueba)} casos correctos.')

## ⚠️ CELDA DE INSPECCIÓN — Revisar antes de filtrar

Muestra la lista completa de títulos que se eliminarían. **No modifica nada.**
Revisá que los resultados sean razonables antes de ejecutar la celda de aplicación.

In [ ]:
# Lista de títulos únicos que serían descartados
titulos_unicos = df['titulo_base'].dropna().unique()
titulos_a_filtrar = sorted([t for t in titulos_unicos if es_sin_valor(t)])

# Conteo de filas por título
conteo = df[df['titulo_base'].isin(titulos_a_filtrar)].groupby('titulo_base').size().rename('filas')

print(f'Títulos únicos en el dataset:       {len(titulos_unicos)}')
print(f'Títulos que se filtrarían:          {len(titulos_a_filtrar)}')
print(f'Filas que se eliminarían:           {conteo.sum()}')
print()

# Verificar que los 5 borderline CON DESCRIPCIÓN no están en la lista de filtrado.
# (El sexto caso borderline, Exp. 7835-D-00 - pliegos de magistrados, SÍ se filtra
# intencionalmente porque son designaciones de personas, no leyes temáticas.)
borderline_conservar = [
    'Expte. 0089-S-2020. DE LEY. CONVENIO DE TRANSFERENCIA PROGRESIVA A CABA',
    'Expte. 0366-D-2020. DE LEY. PROMOCION DEL USO DE CISTERNAS DE DOBLE DESCARGA EN INSTALACIONES SANITARIAS',
    'Exp. 498-D-21 - Ayuda Transportistas Escolares',
    'Exp. 581-D-2009 - Sistema de Faros Centenarios',
    'Exp. 2183-D-09 -  Impuestos Internos',
]
problemas = [t for t in borderline_conservar if t in titulos_a_filtrar]
if problemas:
    print('⚠️  PROBLEMA: estos títulos están siendo filtrados incorrectamente:')
    for t in problemas:
        print(f'   - {t}')
else:
    print('✓ Los 5 títulos borderline con descripción NO están en la lista de filtrado.')

print('✓ Exp. 7835-D-00 (pliegos magistrados) SÍ se filtra (decisión de diseño).')
print()

# Mostrar lista completa con conteo de filas
print('─' * 80)
print(f'{"TÍTULO":<65} {"FILAS":>5}')
print('─' * 80)
for titulo in titulos_a_filtrar:
    n = conteo.get(titulo, 0)
    print(f'{titulo[:65]:<65} {n:>5}')

## ⚠️ CELDA DE APLICACIÓN — Ejecutar solo después de revisar la inspección

Aplica el filtro y guarda `df_modelado.csv`. No modifica `df_consolidado.csv`.

In [ ]:
filas_antes = len(df)
titulos_antes = df['titulo_base'].nunique()

# Aplicar filtro: conservar solo filas cuyo titulo_base NO está en la lista a filtrar
df_modelado = df[~df['titulo_base'].isin(titulos_a_filtrar)].copy()

filas_despues = len(df_modelado)
titulos_despues = df_modelado['titulo_base'].nunique()

print('─' * 50)
print('RESUMEN DEL FILTRADO')
print('─' * 50)
print(f'Títulos únicos eliminados:  {titulos_antes - titulos_despues}')
print(f'Filas eliminadas:           {filas_antes - filas_despues}')
print(f'Filas restantes:            {filas_despues}')
print(f'Títulos únicos restantes:   {titulos_despues}')
print()

# Guardar — nombre distinto para no sobreescribir el original
ruta_salida = '../data/df_modelado.csv'
df_modelado.to_csv(ruta_salida, index=False)
print(f'Guardado en: {ruta_salida}')

# Verificar que df_consolidado no fue modificado
df_original = pd.read_csv('../data/df_consolidado.csv')
assert len(df_original) == filas_antes, '⚠️ ERROR: df_consolidado.csv fue modificado'
print('✓ df_consolidado.csv intacto (misma cantidad de filas que al inicio).')